# Data splits

## 1. Import libraries

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

## 2. Load data

In [2]:
df = pd.read_csv("..\\dataset\\fragments_metadata_filtered.csv")
df

,id,name,age,gender,position,record_id,segment,label,category,duration
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525
4,9,P3,4.3,1,p1,24116,0,Normal,Normal,2.40825
...,...,...,...,...,...,...,...,...,...,...
20866,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025
20867,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525
20868,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525
20869,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925


## 3. Train-test split

### 3.1. Train/Validation-Test sets

In [3]:
X = df["id"].values
y = df["label"].values
groups = df["name"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Split
train_val_idx, test_idx = next(sgkf.split(X, y, groups))

df_train_val = df.iloc[train_val_idx].copy()
df_test = df.iloc[test_idx].copy()

In [4]:
print(f"Train/Val: {len(df_train_val)} ({len(df_train_val) / len(df) * 100:.2f}%)")
print(f"Test: {len(df_test)} ({len(df_test) / len(df) * 100:.2f}%)")

print("\nDistribución train/val:")
print(df_train_val["label"].value_counts(normalize=True))

print("\nDistribución test:")
print(df_test["label"].value_counts(normalize=True))

Train/Val: 16697 (80.00%)
Test: 4174 (20.00%)

Distribución train/val:
label
Normal     0.899383
Wheeze     0.086602
Rhonchi    0.010361
Stridor    0.003653
Name: proportion, dtype: float64

Distribución test:
label
Normal     0.899617
Wheeze     0.086727
Rhonchi    0.010541
Stridor    0.003115
Name: proportion, dtype: float64


### 3.2. Train-Validation sets

In [5]:
X_tv = df_train_val["id"].values
y_tv = df_train_val["label"].values
groups_tv = df_train_val["name"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

df_train_val["fold"] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(X_tv, y_tv, groups_tv), start=1):
    df_train_val.iloc[val_idx, df_train_val.columns.get_loc("fold")] = fold

In [6]:
print("Number of samples per fold:")
print(df_train_val["fold"].value_counts().sort_index())

for fold in sorted(df_train_val["fold"].unique()):
    val = df_train_val[df_train_val["fold"] == fold]
    train = df_train_val[df_train_val["fold"] != fold]

    print(f"\nFold {fold}")
    print("Train samples:", len(train), f"({len(train)/len(df_train_val)*100:.2f}%)", "Val samples:", len(val), f"({len(val)/len(df_train_val)*100:.2f}%)")
    print("Train patients:", train["name"].nunique(), "Val patients:", val["name"].nunique())
    print("Train label distribution:")
    print(train["label"].value_counts(normalize=True))
    print("Val label distribution:")
    print(val["label"].value_counts(normalize=True))

Number of samples per fold:
fold
1    3339
2    3337
3    3343
4    3339
5    3339
Name: count, dtype: int64

Fold 1
Train samples: 13358 (80.00%) Val samples: 3339 (20.00%)
Train patients: 589 Val patients: 145
Train label distribution:
label
Normal     0.899386
Wheeze     0.086615
Rhonchi    0.010481
Stridor    0.003518
Name: proportion, dtype: float64
Val label distribution:
label
Normal     0.899371
Wheeze     0.086553
Rhonchi    0.009883
Stridor    0.004193
Name: proportion, dtype: float64

Fold 2
Train samples: 13360 (80.01%) Val samples: 3337 (19.99%)
Train patients: 586 Val patients: 148
Train label distribution:
label
Normal     0.899177
Wheeze     0.086527
Rhonchi    0.010329
Stridor    0.003967
Name: proportion, dtype: float64
Val label distribution:
label
Normal     0.900210
Wheeze     0.086904
Rhonchi    0.010488
Stridor    0.002397
Name: proportion, dtype: float64

Fold 3
Train samples: 13354 (79.98%) Val samples: 3343 (20.02%)
Train patients: 587 Val patients: 147
Train 

## 4. Save splits

In [7]:
df_test["fold"] = -1
new_df = pd.concat([df_train_val, df_test]).sort_values("id")
new_df

,id,name,age,gender,position,record_id,segment,label,category,duration,fold
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725,-1
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725,-1
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225,-1
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525,-1
4,9,P3,4.3,1,p1,24116,0,Normal,Normal,2.40825,5
...,...,...,...,...,...,...,...,...,...,...,...
20866,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025,3
20867,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525,3
20868,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525,3
20869,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925,3


In [8]:
new_df.to_csv("splits/splits.csv", index=False)